In [42]:
import requests
import pandas as pd
import os

In [7]:
def request(sport,league,dates): # all lower cases
    base_url=f'https://site.api.espn.com/apis/site/v2/sports/{sport}/{league}/scoreboard'
    params={
        'limit':1000,
        'dates': dates
    }
    response=requests.get(url=base_url,params=params)
    print('Status code:', response.status_code)
    print('URL:', response.url)
    return response.json()

In [3]:
nfl_22=request('football','nfl',2022)

Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=2022


In [4]:
# Get post-season & Super-Bowl dates
reg_start=nfl_22.get('leagues')[0].get('calendar')[1].get('startDate')
reg_end=nfl_22.get('leagues')[0].get('calendar')[1].get('endDate')
post_start=nfl_22.get('leagues')[0].get('calendar')[2].get('startDate')
post_end=nfl_22.get('leagues')[0].get('calendar')[2].get('endDate')
print('Regular Season Start Date:', reg_start)
print('Regular Season End Date:', reg_end)
print('Post Season Start Date:', post_start)
print('Post Season End Date:', post_end)

Regular Season Start Date: 2022-09-08T07:00Z
Regular Season End Date: 2023-01-12T07:59Z
Post Season Start Date: 2023-01-12T08:00Z
Post Season End Date: 2023-02-15T07:59Z


In [5]:
reg_22=request('football','nfl','20220908-20230112')

Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=20220908-20230112


In [6]:
len(reg_22.get('events'))

271

In [7]:
post_22=request('football','nfl','20230112-20230215')
len(post_22.get('events'))

Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=20230112-20230215


14

In [9]:
nfl_2223=[]
for i in range (0,len(reg_22.get('events'))):
    year=reg_22.get('events')[i].get('season').get('year')
    season_type=reg_22.get('events')[i].get('season').get('type')
    short_name=reg_22.get('events')[i].get('shortName')
    id= f'{year}_{season_type}_{short_name}'
    week=reg_22.get('events')[i].get('week').get('number')
    date=reg_22.get('events')[i].get('competitions')[0].get('date')
    attendance=reg_22.get('events')[i].get('competitions')[0].get('attendance')
    home=reg_22.get('events')[i].get('competitions')[0].get('competitors')[0].get('team').get('name')
    home_score=reg_22.get('events')[i].get('competitions')[0].get('competitors')[0].get('score')
    away=reg_22.get('events')[i].get('competitions')[0].get('competitors')[1].get('team').get('name')
    away_score=reg_22.get('events')[i].get('competitions')[0].get('competitors')[1].get('score')
    home_winner=reg_22.get('events')[i].get('competitions')[0].get('competitors')[0].get('winner')
    away_winner=reg_22.get('events')[i].get('competitions')[0].get('competitors')[1].get('winner')
    if home_winner:
        winner=home
    elif away_winner:
        winner=away
    else:
        winner='Tie'
    nfl_2223.append([id,year,season_type,week,date,attendance,home,away,home_score,away_score,winner])


In [11]:
pd.DataFrame(nfl_2223,columns=['id','year','season type','week','date','attendance','home team','away team','home score','away score','winner'])

,id,year,season type,week,date,attendance,home team,away team,home score,away score,winner
0,2022_2_BUF @ LAR,2022,2,1,2022-09-09T00:20Z,73846,Rams,Bills,10,31,Bills
1,2022_2_NO @ ATL,2022,2,1,2022-09-11T17:00Z,70078,Falcons,Saints,26,27,Saints
2,2022_2_SF @ CHI,2022,2,1,2022-09-11T17:00Z,62159,Bears,49ers,19,10,Bears
3,2022_2_PIT @ CIN,2022,2,1,2022-09-11T17:00Z,65841,Bengals,Steelers,20,23,Steelers
4,2022_2_PHI @ DET,2022,2,1,2022-09-11T17:00Z,64537,Lions,Eagles,35,38,Eagles
...,...,...,...,...,...,...,...,...,...,...,...
266,2022_2_NYG @ PHI,2022,2,18,2023-01-08T21:25Z,69879,Eagles,Giants,22,16,Eagles
267,2022_2_ARI @ SF,2022,2,18,2023-01-08T21:25Z,71638,49ers,Cardinals,38,13,49ers
268,2022_2_LAR @ SEA,2022,2,18,2023-01-08T21:25Z,68660,Seahawks,Rams,19,16,Seahawks
269,2022_2_DAL @ WSH,2022,2,18,2023-01-08T21:25Z,62814,Commanders,Cowboys,26,6,Commanders


In [8]:
def get_dates(sport,league,seasons):
    for season in seasons:
        bang=request(sport,league,season)
        reg_start=bang.get('leagues')[0].get('calendar')[1].get('startDate')
        reg_end=bang.get('leagues')[0].get('calendar')[1].get('endDate')
        post_start=bang.get('leagues')[0].get('calendar')[2].get('startDate')
        post_end=bang.get('leagues')[0].get('calendar')[2].get('endDate')
        print('Regular Season Dates:', f'{reg_start}-{reg_end}')
        print('Post Season Dates:', f'{post_start}-{post_end}')

In [43]:
def pull_data(sport,league,date_ranges,output_file='fantasy_last3yr.xlsx'):

    columns=['id','year','season type','week','date','attendance','home team','away team','home score','away score','winner']
    last3=[]
    for r in date_ranges:
        scoreboard=request(sport,league,r)
        for i in range(0,len(scoreboard.get('events'))):
            year=scoreboard.get('events')[i].get('season').get('year')
            season_type=scoreboard.get('events')[i].get('season').get('type')
            short_name=scoreboard.get('events')[i].get('shortName')
            id= f'{year}_{season_type}_{short_name}'
            if scoreboard.get('events')[i].get('week'):
                week=scoreboard.get('events')[i].get('week').get('number')
            else:
                week=None
            date=scoreboard.get('events')[i].get('competitions')[0].get('date')
            attendance=scoreboard.get('events')[i].get('competitions')[0].get('attendance')
            home=scoreboard.get('events')[i].get('competitions')[0].get('competitors')[0].get('team').get('name')
            home_score=scoreboard.get('events')[i].get('competitions')[0].get('competitors')[0].get('score')
            away=scoreboard.get('events')[i].get('competitions')[0].get('competitors')[1].get('team').get('name')
            away_score=scoreboard.get('events')[i].get('competitions')[0].get('competitors')[1].get('score')
            home_winner=scoreboard.get('events')[i].get('competitions')[0].get('competitors')[0].get('winner')
            away_winner=scoreboard.get('events')[i].get('competitions')[0].get('competitors')[1].get('winner')
            if home_winner:
                winner=home
            elif away_winner:
                winner=away
            else:
                winner='Tie'
            last3.append([id,year,season_type,week,date,attendance,home,away,home_score,away_score,winner])
    last3_df=pd.DataFrame(last3,columns=columns)
    if not os.path.exists(output_file):
        last3_df.to_excel(output_file,sheet_name=f'{league}',index=False)
        print("Created new Excel file.")
    if os.path.exists(output_file):
        with pd.ExcelWriter(output_file,engine='openpyxl',mode='a',if_sheet_exists='replace') as writer:
            last3_df.to_excel(writer,sheet_name=f'{league}',index=False)

        

In [ ]:
# Get dates for nfl (only works for nfl)
get_dates('football','nfl',[2022,2023,2024])

Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=2022
Regular Season Dates: 2022-09-08T07:00Z-2023-01-12T07:59Z
Post Season Dates: 2023-01-12T08:00Z-2023-02-15T07:59Z
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=2023
Regular Season Dates: 2023-09-07T07:00Z-2024-01-11T07:59Z
Post Season Dates: 2024-01-11T08:00Z-2024-02-15T07:59Z
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=2024
Regular Season Dates: 2024-09-05T07:00Z-2025-01-08T07:59Z
Post Season Dates: 2025-01-08T08:00Z-2025-02-15T07:59Z


In [ ]:
# Create dates input to pull data
nfl_dates=['20220908-20230112','20230112-20230215','20230907-20240111','20240111-20240215','20240905-20250108','20250108-20250215']

In [36]:
# Pull nfl data for the last 3 years
pull_data('football','nfl',nfl_dates)

Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=20220908-20230112
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=20230112-20230215
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=20230907-20240111
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=20240111-20240215
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=20240905-20250108
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?limit=1000&dates=20250108-20250215


---
`get_date` function only works for nfl so let's get the dates for three other sports manually

In [ ]:
# We can call the json file here or web search
#request('basketball','nba',2022)

In [22]:
nba_dates=['20221018-20230409','20230411-20230612','20231024-20240414','20240416-20240617','20241022-20250413','20250415-20250622']

In [37]:
pull_data('basketball','nba',nba_dates)

Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?limit=1000&dates=20221018-20230409
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?limit=1000&dates=20230411-20230612
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?limit=1000&dates=20231024-20240414
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?limit=1000&dates=20240416-20240617
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?limit=1000&dates=20241022-20250413
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?limit=1000&dates=20250415-20250622


In [38]:
mlb_dates=['20230330-20231001','20231003-20231101','20240320-20240929','20241001-20241102','20250318-20250928','20250930-20251101']

In [39]:
pull_data('baseball','mlb',mlb_dates)

Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?limit=1000&dates=20230330-20231001
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?limit=1000&dates=20231003-20231101
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?limit=1000&dates=20240320-20240929
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?limit=1000&dates=20241001-20241102
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?limit=1000&dates=20250318-20250928
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?limit=1000&dates=20250930-20251101


In [40]:
nhl_dates=['20221007-20230413','20230417-20230613','20231010-20240418','20240420-20240624','20241004-20250417','20250419-20250617']

In [41]:
pull_data('hockey','nhl',nhl_dates)

Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard?limit=1000&dates=20221007-20230413
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard?limit=1000&dates=20230417-20230613
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard?limit=1000&dates=20231010-20240418
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard?limit=1000&dates=20240420-20240624
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard?limit=1000&dates=20241004-20250417
Status code: 200
URL: https://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard?limit=1000&dates=20250419-20250617
